In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesRegressor
from xgboost import XGBRegressor
import numpy as np
from sklearn.linear_model import Ridge
import shap
import plotly.graph_objects as go
import joblib

In [2]:
RANDOM_STATE = 101
DATE_STR = "2026-04-14"
FEATURES_PATH = f"../data/processed/features_nlp_{DATE_STR}.csv"
MODEL_PATH = "../models/salary_model.pkl"
PRED_PATH = f"../data/processed/unlabeled_salary_predictions_{DATE_STR}.csv"

In [3]:
df_model = pd.read_csv(FEATURES_PATH)

print(f"df_model {df_model.shape}")

df_model (1176, 108)


In [4]:
NO_SKILL = {
    "id", "area", "employment" , "experience", "has_usd_in_description", "salary_mid", 
    "grade", "wf_ON_SITE", "wf_REMOTE", "wf_HYBRID", "wf_FIELD_WORK"
}

SKILLS_COL = [col for col in df_model.columns if col not in NO_SKILL]

# I decided to drop skill columns with < 5% non-zero values because my models worked purely
SKILL_MIN_FREQ = 0.05
SKILLS_COL = [col for col in SKILLS_COL if df_model[col].mean() >= SKILL_MIN_FREQ]

df_model["skills_count"] = df_model[SKILLS_COL].sum(axis=1)

print(
    "skills_count - min/mean/max:",
    df_model["skills_count"].min(),
    round(df_model["skills_count"].mean(), 2),
    df_model["skills_count"].max()
)
print(f"skill columns kept: {len(SKILLS_COL)}")

skills_count - min/mean/max: 0 3.36 18
skill columns kept: 27


In [5]:
FEATURE_COL = [col for col in df_model.columns if col not in {"id", "salary_mid"}]
TARGET = 'salary_mid'

In [6]:
GRADE_MAP = {"Unknown": 0, "Junior": 1, "Middle": 2, "Senior": 3}
df_model["grade"] = df_model["grade"].map(GRADE_MAP).fillna(0).astype(int)

In [7]:
df_labeled = df_model[df_model[TARGET].notna()].copy().reset_index(drop=True)
df_unlabeled = df_model[df_model[TARGET].isna()].copy().reset_index(drop=True)

In [8]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

CAT_COL = ["area", "employment", 'experience']
PASSTHROUGH_COL = [col for col in FEATURE_COL if col not in CAT_COL]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, CAT_COL),
        ('num', "passthrough", PASSTHROUGH_COL)
    ],
    remainder="drop"
)

preprocessor_ridge = ColumnTransformer(
    transformers=[
        ("cat", ohe, CAT_COL),
        ("num", StandardScaler(), PASSTHROUGH_COL)
    ],
    remainder="drop"
)

In [9]:
x = df_labeled[FEATURE_COL]
y = df_labeled[TARGET]

In [10]:
cv = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)

In [11]:
# Firstly test 3 models with same preprocessor in pipeline
models = {
    "Ridge": TransformedTargetRegressor(
        regressor=Pipeline([
            ('pre', preprocessor_ridge),
            ('model', Ridge(alpha=100.0))
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    ),

    # parameters from a small optuna search
    'xgb': TransformedTargetRegressor(
        regressor=Pipeline([
            ("pre", preprocessor),
            ("model", XGBRegressor(
                n_estimators=427,
                max_depth=3,
                subsample=0.8679712364244379,
                learning_rate=0.013840304279246962,
                colsample_bytree=0.6648534468005024,
                min_child_weight=13,
                reg_alpha=0.6409312184006158,
                reg_lambda=4.549266974796569,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    ),

    "et": TransformedTargetRegressor(
        regressor=Pipeline([
            ("pre", preprocessor),
            ("model", ExtraTreesRegressor(
                n_estimators=500,
                max_features="sqrt",
                min_samples_leaf=5,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ]),
        func=np.log1p,
        inverse_func=np.expm1
    ),
}

In [12]:
results = []

for name, pipe in models.items():
    cv_result = cross_validate(
        pipe, x, y, cv=cv,
        scoring={"r2": "r2", "mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error"},
        n_jobs=-1
    )

    mae = -cv_result["test_mae"].mean()
    rmse = -cv_result["test_rmse"].mean()
    r2 = cv_result["test_r2"].mean()
    r2_std = cv_result["test_r2"].std()

    results.append({"model": name, "MAE": mae, "RMSE": rmse, "R2 score": r2, "CV R2 std": r2_std})
    print(f"{name}: MAE={mae:_.2f}, RMSE={rmse:_.2f}, R2={r2:.3f} ± {r2_std:.3f}")

Ridge: MAE=291_366.61, RMSE=459_823.12, R2=0.184 ± 0.118
xgb: MAE=265_656.33, RMSE=417_003.34, R2=0.330 ± 0.098
et: MAE=287_343.89, RMSE=461_117.84, R2=0.188 ± 0.059


In [13]:
#

yeah, not ideal. but with only 367 labeled rows and salary variance this high,
CV R2=0.330 is honestly the best i could get out here

 **XGBoost** as best model will predict unlabeled salaries

In [14]:
# now use everything we have
best_model = models["xgb"]
best_model.fit(x, y)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","Pipeline(step...=None, ...))])"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed 

In [15]:
X_unlabeled = df_unlabeled[FEATURE_COL]
salary_preds = best_model.predict(X_unlabeled)

out = df_unlabeled[["id", "area", "employment", "grade"]].copy()
out["salary_pred"] = salary_preds.round(0).astype(int)

out.to_csv(PRED_PATH, index=False)

print(f"predicted salaries for {len(out)} unlabeled vacancies")
print(f"saved to: {PRED_PATH}")
print(out["salary_pred"].describe())

predicted salaries for 809 unlabeled vacancies
saved to: ../data/processed/unlabeled_salary_predictions_2026-04-14.csv
count    8.090000e+02
mean     7.184294e+05
std      2.875368e+05
min      2.663910e+05
25%      5.027710e+05
50%      6.817820e+05
75%      8.425530e+05
max      1.958024e+06
Name: salary_pred, dtype: float64


In [16]:
# shap setup
inner_pipe = best_model.regressor_

feature_names = list(inner_pipe.named_steps["pre"].get_feature_names_out())

print(f"sample: {feature_names[:8]}")

sample: ['cat__area_Актау', 'cat__area_Актобе', 'cat__area_Алматы', 'cat__area_Астана', 'cat__area_Атырау', 'cat__area_Боралдай', 'cat__area_Жезказган', 'cat__area_Караганда']


In [17]:
SHAP_SAMPLE = min(300, len(x))
X_sample = x.sample(SHAP_SAMPLE, random_state=RANDOM_STATE)

X_sample_transformed = inner_pipe.named_steps["pre"].transform(X_sample)

explainer = shap.TreeExplainer(inner_pipe.named_steps["model"])
shap_result = explainer(X_sample_transformed, check_additivity=False)

shap_vals = shap_result.values
baseline_log = float(shap_result.base_values.mean())

In [18]:
shap_df = pd.DataFrame(shap_vals, columns=feature_names)

skill_feature_names = [f"num__{s}" for s in SKILLS_COL]
missing_skill_features = [f for f in skill_feature_names if f not in shap_df.columns]
if missing_skill_features:
    print("some skill features are missing after preprocessing:")
    print(missing_skill_features[:10])

skill_feature_names = [f for f in skill_feature_names if f in shap_df.columns]
skill_names = [f.replace("num__", "", 1) for f in skill_feature_names]
skill_shap_vals = shap_df[skill_feature_names].values  # (n_samples, n_skills)

log_preds = baseline_log + shap_vals.sum(axis=1, keepdims=True)
salary_ctx = np.expm1(log_preds)
impact_matrix = salary_ctx - np.expm1(log_preds - skill_shap_vals)   # (n, n_skills)

In [19]:
skill_impact = pd.DataFrame({
    "skill": skill_names,
    "mean_shap_log": shap_df[skill_feature_names].mean().values,
    "mean_abs_shap_log": shap_df[skill_feature_names].abs().mean().values,
    "approx_kzt_shift": impact_matrix.mean(axis=0),
    "approx_abs_kzt_shift": np.abs(impact_matrix).mean(axis=0),
})

skill_impact = skill_impact.sort_values("mean_shap_log", ascending=False).reset_index(drop=True)

print("top 10 salary-increasing skills:")
print(skill_impact[["skill", "mean_shap_log", "approx_kzt_shift"]]
    .head(10).to_string(index=False))

print("\ntop 10 salary-decreasing skills:")
print(skill_impact[["skill", "mean_shap_log", "approx_kzt_shift"]]
    .tail(10).sort_values("mean_shap_log").to_string(index=False))

top 10 salary-increasing skills:
     skill  mean_shap_log  approx_kzt_shift
      REST       0.001180       1185.722290
       Git       0.001141        835.178223
  RabbitMQ       0.000608       1300.647461
       SQL       0.000482        724.612061
     React       0.000407        -83.456612
     CI/CD       0.000357       3323.372314
      Jira       0.000147        122.949997
PostgreSQL       0.000126       -225.073288
TypeScript       0.000088         78.205986
        1C       0.000049        119.813850

top 10 salary-decreasing skills:
     skill  mean_shap_log  approx_kzt_shift
    Python      -0.003076       3770.817139
JavaScript      -0.000648       -676.000854
      Java      -0.000459        490.384277
Kubernetes      -0.000096         20.112709
    Docker      -0.000003        159.547546
Prometheus       0.000000          0.000000
     MySQL       0.000000          0.000000
       AWS       0.000000          0.000000
    Oracle       0.000000          0.000000
      Bas

In [20]:
top_abs = skill_impact.nlargest(20, "approx_abs_kzt_shift")

print("top 20 skills by absolute SHAP impact:")
print(top_abs[["skill", "mean_shap_log", "mean_abs_shap_log", "approx_abs_kzt_shift"]].to_string(index=False))

top 20 skills by absolute SHAP impact:
     skill  mean_shap_log  mean_abs_shap_log  approx_abs_kzt_shift
    Python      -0.003076           0.054173          35626.160156
     CI/CD       0.000357           0.026728          17295.000000
      REST       0.001180           0.018250          12867.243164
       SQL       0.000482           0.018944          11461.182617
JavaScript      -0.000648           0.016937          11088.766602
     React       0.000407           0.008526           5838.134766
      Java      -0.000459           0.006973           4828.841797
       Git       0.001141           0.006818           4780.819824
  RabbitMQ       0.000608           0.006183           4459.560059
PostgreSQL       0.000126           0.003895           2892.111328
    Docker      -0.000003           0.001683           1096.411011
      Jira       0.000147           0.001851           1052.532959
        1C       0.000049           0.001747           1043.282593
     Linux       0.0000

In [21]:
# top skills that add salary
_dark = dict(template="plotly_dark", width=1000, height=500)

top_pos = skill_impact.nlargest(15, "approx_kzt_shift")

fig1 = go.Figure(go.Bar(
    x=top_pos["approx_kzt_shift"],
    y=top_pos["skill"],
    orientation="h",
    marker_color="#4ade80",
    text=top_pos["approx_kzt_shift"].apply(lambda v: f"+ {v:,.0f} ₸"),
    textposition="outside",
))

fig1.update_layout(
    title="Skills that boost salary most",
    xaxis_title="Salary impact, KZT",
    yaxis_title="",
    margin=dict(l=140, r=60),
    **_dark,
)
fig1.show()

In [22]:
# top skills that decrease salary
top_neg = (skill_impact[skill_impact["approx_kzt_shift"] < 0].nsmallest(15, "approx_kzt_shift")
           .sort_values("approx_kzt_shift", ascending=False))
fig2 = go.Figure(go.Bar(
    x=top_neg["approx_kzt_shift"],
    y=top_neg["skill"],
    orientation="h",
    marker_color="#f87171",
    text=top_neg["approx_kzt_shift"].apply(lambda v: f"{v:,.0f} ₸"),
    textposition="outside",
))

fig2.update_layout(
    title="Skills that decrease salary most",
    xaxis_title="Salary impact, KZT",
    yaxis_title="",
    margin=dict(l=140, r=60),
    **_dark,
)
fig2.show()

In [23]:
# top 20 skills by absolute impact
top_abs_plot = top_abs.sort_values("approx_abs_kzt_shift")

colors = ["#4ade80" if v >= 0 else "#f87171" for v in top_abs_plot["approx_kzt_shift"]]

fig3 = go.Figure(go.Bar(
    x=top_abs_plot["approx_abs_kzt_shift"],
    y=top_abs_plot["skill"],
    orientation="h",
    marker_color=colors,
    text=top_abs_plot["approx_abs_kzt_shift"].apply(lambda v: f"{v:,.0f} ₸"),
    textposition="outside",
))

fig3.update_layout(
    title="Top 20 skills by absolute salary impact",
    xaxis_title="Salary |impact|, KZT",
    yaxis_title="",
    margin=dict(l=140, r=80),
    **_dark,
)
fig3.show()

In [24]:
joblib.dump(best_model, MODEL_PATH)

['../models/salary_model.pkl']